# Modelo realista · mejoras de Adri sobre `real_todo` de Pablo (`realista/`)

**Qué es:** el LightGBM realista de Pablo (`realista/m_lgb_real.py`): ve el **snapshot actual y snapshots PASADOS** de la
sesión, como funcionaría en tiempo real (nada del futuro, nada de etiquetas de otros controladores). Su mejor versión,
`real_todo` (bloques `base + logic + past + causal + geo`, 3 semillas), da **CV 0.6703** y **0.6718** con el segundo paso causal.

**Qué se añade aquí** (lo que funcionó en `notebooks/LGB_adri.ipynb` con el modelo de un solo snapshot):

| Idea | De dónde | Resultado allí |
|---|---|---|
| **Peso por tramo**: al entrenar, cada snapshot pesa 1/√(longitud de su tramo) | `LGB_adri.ipynb`, 12c–12d | **+0.8**, robusto (3 escenarios, 24 de 37 sesiones suben, peor controlador +0.8) |
| **10 semillas** | futuro 9b, `LGB_adri` 12d | +0.1–0.2 |
| **Quitar las variables que delatan al controlador** (lista de Juanma) | `pruebas/notas/h4_sel.md` | +0.4, peor controlador +1.4 (sin peso; nunca combinado con él) |

**Reglas:** mismas features, folds (`Data/folds.csv`) y parámetros que Pablo (reutiliza su código sin modificarlo). El peso usa
la etiqueta **solo de las filas de entrenamiento**: la predicción de cada snapshot sigue usando solo ese snapshot y los anteriores.

**Cómo usarlo:** ejecuta *Preparación* (~3–5 min: calcula las features de Pablo) y luego los bloques en orden. Cada experimento
se apunta en `experiments/lgb_real_runs.csv` (el registro de Pablo) con el prefijo `adri_`. Se compara siempre contra la base
**con las mismas semillas** (bloque 0), con las métricas robustas de `futuro/metricas.py`: CV, peor controlador, mediana por
sesión, sesiones y controladores que suben / bajan.

> ## Punto de partida actual (21/10, 02:45)
>
> **Mejor modelo realista: `real_todo` (Pablo) = CV 0.6703, 0.6718 con segundo paso causal**, peor controlador 0.600.
>
> | Paso | CV | peor controlador | estado | dónde |
> |---|---|---|---|---|
> | `real_todo` (Pablo, 3 semillas) | 0.6703 | 0.596 | referencia | `LGB_realista.ipynb` |
> | + peso por tramo | | | **siguiente** | 1 |
> | + quitar variables que delatan al controlador | | | **siguiente** | 2 |
> | versión final, 10 semillas + segundo paso causal | | | **siguiente** | 3 |

## Preparación

In [ ]:
import json, os, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

# Los scripts de Pablo importan fcntl (bloqueo de archivos de Linux), que no existe en Windows: sustituto que no hace nada.
sys.modules.setdefault("fcntl", types.SimpleNamespace(flock=lambda *a, **k: None, LOCK_EX=2))

ROOT = Path.cwd().parent if Path.cwd().name in ("realista", "notebooks") else Path.cwd()
REAL = ROOT / "realista"
for p in ("cv", "futuro", "realista"):
    sys.path.insert(0, str(ROOT / p))
import m_lgb_real as R, m_lgb_future as M, feat_session_causal, metricas
from metric import save_outputs

data = R.build_data()                       # features de Pablo (~3-5 min la primera vez)
lab, te, fs = data["lab"], data["te"], data["fs"]
M.add_lengths(lab)                          # n_tramo, n_racha: solo para pesar el entrenamiento
_, P = R.parse(["--threads", str(os.cpu_count() or 8)])   # parámetros de Pablo (los de EQUIPO.md)
FEATS = "base,logic,past,causal,geo"       # los bloques de real_todo
print("bloques:", {k: len(v) for k, v in fs.items()})

In [ ]:
OOF = {}                                    # nombre -> oof (sample_id,row_index,score)


def cv_real(nombre, feats=FEATS, peso=None, pot=0.5, drop=(), seeds=2, ref=None, guardar=False, sesion=False, tag=""):
    """CV con los folds del equipo. peso: None, "tramo" o "racha" (1/longitud^pot al entrenar). drop: columnas a quitar.
    ref: nombre de otro experimento de OOF para comparar. guardar: entrena con todo y guarda en realista/oof|preds|submissions."""
    t0 = time.time()
    F = [c for b in feats.split(",") for c in fs[b] if c not in set(drop)]
    w = None if peso is None else 1.0 / lab["n_" + peso].values ** pot
    grp = lambda A: A.groupby("sample_id", sort=False).size().values
    oof = np.zeros(len(lab))
    for f in sorted(lab.fold.unique()):
        tr, va = (lab.fold != f).values, (lab.fold == f).values
        A = lab[tr]
        for s in range(seeds):
            m = M.model(P, s).fit(A[F], A.target_index, group=grp(A), sample_weight=None if w is None else w[tr],
                                  categorical_feature=["task_index"])
            oof[va] += m.predict(lab.loc[va, F]) / seeds
    oof_df = lab[["sample_id", "row_index"]].assign(score=oof)
    OOF[nombre] = oof_df
    met = metricas.metricas(oof_df, OOF.get(ref))
    print(metricas.resumen(nombre, met, ref), f"  [{len(F)} features, {time.time() - t0:.0f}s]", flush=True)
    a = types.SimpleNamespace(tag=tag, feats=feats, drop=",".join(drop), seeds=seeds, sesion=sesion)
    R.log_run(a, nombre, met, len(F), time.time() - t0)
    if guardar:
        pred = np.mean([M.model(P, s).fit(lab[F], lab.target_index, group=grp(lab), sample_weight=w,
                                          categorical_feature=["task_index"]).predict(te[F]) for s in range(seeds)], axis=0)
        save_outputs(nombre, oof_df, te[["sample_id", "row_index"]].assign(score=pred), out=REAL)
        if sesion:
            so, sp, pesos = feat_session_causal.sesion(lab, oof, te, pred)
            o2 = lab[["sample_id", "row_index"]].assign(score=so)
            OOF[nombre + "_ses"] = o2
            m2 = metricas.metricas(o2, OOF.get(ref + "_ses") if ref else None)
            print(metricas.resumen(nombre + "_ses", m2, ref + "_ses" if ref else None), flush=True)
            print("  pesos del segundo paso causal:", pesos)
            save_outputs(nombre + "_ses", o2, te[["sample_id", "row_index"]].assign(score=sp), out=REAL)
            R.log_run(a, nombre + "_ses", m2, len(F), time.time() - t0)
    return met


def tabla(nombres, ref):
    """Tabla resumen frente a ref (mismas semillas)."""
    filas = []
    for n in nombres:
        m = metricas.metricas(OOF[n], OOF[ref] if n != ref else None)
        filas.append(dict(modelo=n, cv=m["cv"], peor_atco=m["min_atco"], mediana_sesion=m["ses_median"],
                          esc1=m.get("sc1"), esc2=m.get("sc2"), esc3=m.get("sc3"),
                          d_cv=m.get("d_cv"), sesiones_suben=m.get("ses_up"), sesiones_bajan=m.get("ses_down"),
                          controladores_suben=m.get("atco_up"), controladores_bajan=m.get("atco_down")))
    return pd.DataFrame(filas).round(4)

## 0 · Punto de partida con 2 semillas

`real_todo` de Pablo, pero con **2 semillas** (las mismas que usarán las variantes), para que las comparaciones sean justas.
Pablo dio 0.6703 con 3. ~3 min.

In [ ]:
cv_real("adri_base", tag="real_todo con 2 semillas (referencia)")

## 1 · Peso por tramo

**Hipótesis:** en `LGB_adri.ipynb` (solo snapshot), que los tramos largos (muchos snapshots casi iguales con la misma respuesta)
no dominen el entrenamiento subió +0.8 de forma robusta. Con el pasado debería pasar lo mismo, o más: las features de pasado
hacen que los snapshots de un tramo se parezcan todavía más entre sí.

| Variante | Qué cambia | Por qué |
|---|---|---|
| `adri_tramo` | peso 1/√(longitud del tramo) | lo que ganó en `LGB_adri` (12c, 12d) |
| `adri_racha` | peso 1/√(longitud de la racha), sin mirar la etiqueta | allí empató con el tramo (13a) |

2 semillas, ~3 min por variante.

In [ ]:
cv_real("adri_tramo", peso="tramo", ref="adri_base", tag="+ peso 1/sqrt(tramo)")
cv_real("adri_racha", peso="racha", ref="adri_base", tag="+ peso 1/sqrt(racha)")
tabla(["adri_base", "adri_tramo", "adri_racha"], "adri_base")

**Conclusión bloque 1 · (rellenar tras ejecutar)**

## 2 · Quitar las variables que delatan al controlador

**Hipótesis (Juanma, `pruebas/notas/h4_sel.md`):** un clasificador distingue a cada controlador del resto con AUC 0.999. Lo
delatan sobre todo `a_n300`, `a_n60` y `seq_len` (la carga de tráfico de la sesión) y unas 20 columnas casi inútiles. Sin
peso, quitarlas subió la CV +0.4 y el peor controlador +1.4. El privado es un controlador nuevo: lo que no dependa de su estilo
debería generalizar mejor.

| Variante | Qué se quita (sobre `adri_tramo`) |
|---|---|
| `adri_tramo_sin3` | solo `a_n300`, `a_n60`, `seq_len` |
| `adri_tramo_sel` | las 23 columnas de Juanma (`pruebas/notas/h4_quitar.json`) |

2 semillas, ~3 min por variante.

In [ ]:
QUITAR_JUANMA = json.load(open(ROOT / "pruebas" / "notas" / "h4_quitar.json"))
print("de la lista de Juanma están en estas features:", [c for c in QUITAR_JUANMA if any(c in v for v in fs.values())])
cv_real("adri_tramo_sin3", peso="tramo", drop=["a_n300", "a_n60", "seq_len"], ref="adri_tramo",
        tag="+ peso tramo, sin a_n300/a_n60/seq_len")
cv_real("adri_tramo_sel", peso="tramo", drop=QUITAR_JUANMA, ref="adri_tramo", tag="+ peso tramo, sin las 23 de Juanma")
tabla(["adri_tramo", "adri_tramo_sin3", "adri_tramo_sel"], "adri_tramo")

**Conclusión bloque 2 · (rellenar tras ejecutar)**

## 3 · Versión final: 10 semillas + segundo paso causal

Pon en `PESO_FINAL` y `DROP_FINAL` lo que haya ganado en los bloques 1 y 2. Entrena con **10 semillas**, aplica el segundo paso
causal de Pablo y guarda en `realista/oof|preds|submissions` como **`real_adri`** y **`real_adri_ses`**. ~20 min.
**Solo con `GENERAR_FINAL = True`.** Para comparar, se genera también `real_todo` con 10 semillas (`adri_base10`, sin guardar).

In [ ]:
GENERAR_FINAL = False
PESO_FINAL = "tramo"        # <- "tramo", "racha" o None según el bloque 1
DROP_FINAL = []             # <- ["a_n300", "a_n60", "seq_len"] o QUITAR_JUANMA según el bloque 2

if GENERAR_FINAL:
    cv_real("adri_base10", seeds=10, tag="real_todo con 10 semillas (referencia)")
    cv_real("real_adri", peso=PESO_FINAL, drop=DROP_FINAL, seeds=10, ref="adri_base10", guardar=True, sesion=True,
            tag="versión final de Adri")
    display(tabla(["adri_base10", "real_adri", "real_adri_ses"], "adri_base10"))
    s = pd.read_csv(REAL / "submissions" / "sub_real_adri_ses.csv")
    print("submission real_adri_ses:", len(s), "filas,", s.sample_id.is_unique, "| 3 ids:", s.prediction.str.split().str.len().eq(3).all())
else:
    print("No se genera (GENERAR_FINAL = False).")

**Resultado bloque 3 · (rellenar tras ejecutar)**

| | CV | peor controlador | mediana por sesión | sesiones suben / bajan | Kaggle público |
|---|---|---|---|---|---|
| `real_todo` (Pablo, 10 semillas) | | | | — | |
| `real_adri` | | | | | |
| `real_adri_ses` | | | | | |